# Automated Web Scanner — Secure Headers & CORS Audit

**Program:** IBM SkillsBuild Cyber Security 2026 (BharatCares)
**Project #2:** Automated Web Scanner — a Python script that audits sites for secure HTTP headers and CORS flaws.

### What this notebook does
1. Sends safe, read-only HTTP requests (`GET` / `OPTIONS`) to a target URL.
2. Audits **security response headers** (HSTS, CSP, X-Frame-Options, X-Content-Type-Options, Referrer-Policy, Permissions-Policy, COOP, CORP).
3. Detects **information leakage** (`Server`, `X-Powered-By`) and weak **cookie flags** (`Secure`, `HttpOnly`, `SameSite`).
4. Tests for **CORS misconfigurations** (reflected origin, `null` origin, wildcard + credentials, suffix-trust bypass, permissive preflight).
5. Scores each site (0–100, grade A–F), then exports **JSON / CSV / HTML** reports and a chart.

> ⚠️ **Ethical use:** Only scan systems you own or have written permission to test. This tool is passive (no exploitation, no fuzzing, no brute force), but unauthorised scanning may still be illegal. The built-in demo uses a **local test server** that this notebook starts itself.

In [ ]:
# ---- 1. Imports & configuration ------------------------------------------------
import json, html, csv, threading, time, os
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse

import requests
import pandas as pd
import matplotlib.pyplot as plt

TIMEOUT = 8            # seconds per request
DELAY = 0.3            # polite delay between requests (seconds)
USER_AGENT = "EduWebScanner/1.0 (IBM SkillsBuild project; passive audit)"
REPORT_DIR = "reports"
os.makedirs(REPORT_DIR, exist_ok=True)

SEVERITY_PENALTY = {"High": 15, "Medium": 8, "Low": 3, "Info": 0}
print("Configuration loaded.")

## 2. Security-header rules
Each rule has a **severity**, a plain-English explanation, a **fix**, and a validator that decides whether the header is *present and well configured*, *present but weak*, or *missing*.

In [ ]:
# ---- 2. Header rules -----------------------------------------------------------
def _hsts(v):
    import re
    m = re.search(r"max-age\s*=\s*(\d+)", v, re.I)
    if not m:
        return "WEAK", "max-age directive missing"
    age = int(m.group(1))
    if age < 15552000:                     # 180 days
        return "WEAK", f"max-age={age} is below 180 days"
    return "PASS", f"max-age={age}" + (", includeSubDomains" if "includesubdomains" in v.lower() else "")

def _csp(v):
    low = v.lower()
    problems = []
    if "unsafe-inline" in low: problems.append("allows 'unsafe-inline'")
    if "unsafe-eval" in low:   problems.append("allows 'unsafe-eval'")
    if " * " in f" {low} " or "script-src *" in low or "default-src *" in low:
        problems.append("wildcard (*) source")
    if "default-src" not in low and "script-src" not in low:
        problems.append("no default-src/script-src")
    return ("WEAK", "; ".join(problems)) if problems else ("PASS", "policy present")

def _xfo(v):
    return ("PASS", v) if v.strip().upper() in ("DENY", "SAMEORIGIN") else ("WEAK", f"unexpected value: {v}")

def _xcto(v):
    return ("PASS", v) if v.strip().lower() == "nosniff" else ("WEAK", f"expected 'nosniff', got: {v}")

def _referrer(v):
    bad = {"unsafe-url", "no-referrer-when-downgrade", ""}
    return ("WEAK", f"leaky policy: {v}") if v.strip().lower() in bad else ("PASS", v)

def _present(v):
    return ("PASS", v[:80]) if v.strip() else ("WEAK", "empty value")

HEADER_RULES = {
    "Strict-Transport-Security": dict(severity="High", check=_hsts, https_only=True,
        why="Forces browsers to use HTTPS; prevents SSL-stripping / downgrade attacks.",
        fix="Strict-Transport-Security: max-age=31536000; includeSubDomains"),
    "Content-Security-Policy": dict(severity="High", check=_csp,
        why="Restricts where scripts/styles/frames load from; main defence against XSS.",
        fix="Content-Security-Policy: default-src 'self'; frame-ancestors 'none'"),
    "X-Frame-Options": dict(severity="Medium", check=_xfo,
        why="Stops the page being embedded in a hostile iframe (clickjacking).",
        fix="X-Frame-Options: DENY   (or CSP frame-ancestors)"),
    "X-Content-Type-Options": dict(severity="Medium", check=_xcto,
        why="Disables MIME-type sniffing so uploads cannot be executed as scripts.",
        fix="X-Content-Type-Options: nosniff"),
    "Referrer-Policy": dict(severity="Low", check=_referrer,
        why="Controls how much URL information is leaked to other sites.",
        fix="Referrer-Policy: strict-origin-when-cross-origin"),
    "Permissions-Policy": dict(severity="Low", check=_present,
        why="Limits access to camera, microphone, geolocation and other browser features.",
        fix="Permissions-Policy: geolocation=(), camera=(), microphone=()"),
    "Cross-Origin-Opener-Policy": dict(severity="Low", check=_present,
        why="Isolates the browsing context from cross-origin windows (Spectre-style leaks).",
        fix="Cross-Origin-Opener-Policy: same-origin"),
    "Cross-Origin-Resource-Policy": dict(severity="Low", check=_present,
        why="Prevents other origins from loading your resources.",
        fix="Cross-Origin-Resource-Policy: same-origin"),
}
print(f"{len(HEADER_RULES)} header rules defined.")

## 3. Audit functions
`audit_headers` checks security headers and information leakage, `audit_cookies` checks cookie flags, and `audit_cors` sends requests with crafted `Origin` headers to see how the server responds.

In [ ]:
# ---- 3a. Header, leakage and cookie audits -------------------------------------
LOCAL_HOSTS = {"localhost", "127.0.0.1", "::1"}

def make_finding(category, name, severity, status, detail, fix=""):
    return dict(category=category, check=name, severity=severity,
                status=status, detail=detail, fix=fix)

def audit_headers(resp):
    findings = []
    is_https = resp.url.lower().startswith("https://")
    host = urlparse(resp.url).hostname or ""

    # transport
    if not is_https and host not in LOCAL_HOSTS:
        findings.append(make_finding("Transport", "HTTPS in use", "High", "FAIL",
                        "Site is served over plain HTTP.", "Redirect all traffic to HTTPS."))

    for name, rule in HEADER_RULES.items():
        if rule.get("https_only") and not is_https:
            findings.append(make_finding("Header", name, "Info", "N/A",
                            "Only applies to HTTPS responses."))
            continue
        value = resp.headers.get(name)
        if value is None:
            findings.append(make_finding("Header", name, rule["severity"], "MISSING",
                            rule["why"], rule["fix"]))
        else:
            status, detail = rule["check"](value)
            sev = rule["severity"] if status != "PASS" else "Info"
            findings.append(make_finding("Header", name, sev, status, detail,
                            rule["fix"] if status != "PASS" else ""))

    # information leakage
    server = resp.headers.get("Server", "")
    if any(ch.isdigit() for ch in server):
        findings.append(make_finding("Leakage", "Server banner", "Low", "WEAK",
                        f"Reveals software/version: {server}",
                        "Hide version (e.g. ServerTokens Prod / server_tokens off)."))
    for leak in ("X-Powered-By", "X-AspNet-Version", "X-AspNetMvc-Version"):
        if leak in resp.headers:
            findings.append(make_finding("Leakage", leak, "Low", "WEAK",
                            f"Reveals technology: {resp.headers[leak]}",
                            f"Remove the {leak} header."))
    return findings


def audit_cookies(resp):
    findings = []
    is_https = resp.url.lower().startswith("https://")
    try:
        raw_cookies = resp.raw.headers.getlist("Set-Cookie")
    except AttributeError:
        raw_cookies = []
    for raw in raw_cookies:
        parts = [p.strip() for p in raw.split(";")]
        cname = parts[0].split("=")[0]
        flags = [p.lower() for p in parts[1:]]
        missing = []
        if is_https and "secure" not in flags: missing.append("Secure")
        if "httponly" not in flags: missing.append("HttpOnly")
        if not any(f.startswith("samesite") for f in flags): missing.append("SameSite")
        if missing:
            findings.append(make_finding("Cookie", f"Cookie '{cname}'", "Medium", "WEAK",
                            "Missing flags: " + ", ".join(missing),
                            "Set-Cookie: name=value; Secure; HttpOnly; SameSite=Lax"))
        else:
            findings.append(make_finding("Cookie", f"Cookie '{cname}'", "Info", "PASS",
                            "All recommended flags set."))
    return findings

In [ ]:
# ---- 3b. CORS audit ------------------------------------------------------------
def audit_cors(session, url):
    findings = []
    host = urlparse(url).hostname or "target"
    test_origins = {
        "arbitrary origin":   "https://evil.example",
        "null origin":        "null",
        "suffix-trust bypass": f"https://{host}.evil.example",
    }

    for label, origin in test_origins.items():
        try:
            r = session.get(url, headers={"Origin": origin}, timeout=TIMEOUT)
        except requests.RequestException as e:
            findings.append(make_finding("CORS", label, "Info", "ERROR", str(e)))
            continue
        time.sleep(DELAY)
        acao = r.headers.get("Access-Control-Allow-Origin")
        acac = (r.headers.get("Access-Control-Allow-Credentials", "").lower() == "true")
        vary = r.headers.get("Vary", "").lower()

        if acao is None:
            findings.append(make_finding("CORS", label, "Info", "PASS",
                            "No CORS headers returned for this origin (restrictive)."))
        elif acao == "*":
            if acac:
                findings.append(make_finding("CORS", label, "Medium", "FAIL",
                    "Wildcard '*' combined with Allow-Credentials: true (invalid & risky config).",
                    "Never combine '*' with credentials; use an explicit allow-list."))
            else:
                findings.append(make_finding("CORS", label, "Low", "WEAK",
                    "Wildcard '*' allows any site to read responses (fine only for fully public data).",
                    "Restrict to specific trusted origins if data is not public."))
        elif acao == origin:
            sev = "High" if acac else "Medium"
            what = "null origin" if origin == "null" else "untrusted origin"
            findings.append(make_finding("CORS", label, sev, "FAIL",
                f"Server trusts {what} '{origin}'" + (" WITH credentials" if acac else "") + ".",
                "Validate Origin against a strict allow-list; never reflect it blindly."))
            if "origin" not in vary:
                findings.append(make_finding("CORS", f"{label} - Vary header", "Low", "WEAK",
                    "Origin is reflected but 'Vary: Origin' is missing (cache poisoning risk).",
                    "Add 'Vary: Origin'."))
        else:
            findings.append(make_finding("CORS", label, "Info", "PASS",
                            f"Allows only a fixed origin: {acao}"))

    # Preflight test
    try:
        pf = session.options(url, headers={
            "Origin": "https://evil.example",
            "Access-Control-Request-Method": "PUT",
            "Access-Control-Request-Headers": "Authorization"}, timeout=TIMEOUT)
        methods = pf.headers.get("Access-Control-Allow-Methods", "")
        pf_acao = pf.headers.get("Access-Control-Allow-Origin")
        risky = [m for m in ("PUT", "DELETE", "PATCH") if m in methods.upper()]
        if pf_acao in ("*", "https://evil.example") and (risky or methods.strip() == "*"):
            findings.append(make_finding("CORS", "Preflight policy", "Medium", "FAIL",
                f"Preflight allows untrusted origin with methods: {methods}",
                "Limit Allow-Methods/Allow-Headers and restrict origins."))
        else:
            findings.append(make_finding("CORS", "Preflight policy", "Info", "PASS",
                            "Preflight does not grant risky access to untrusted origins."))
    except requests.RequestException as e:
        findings.append(make_finding("CORS", "Preflight policy", "Info", "ERROR", str(e)))
    return findings

## 4. Scoring and the main `scan_site` function
Score = 100 minus a penalty per failed check (High −15, Medium −8, Low −3). Grades: **A** ≥ 90, **B** ≥ 75, **C** ≥ 60, **D** ≥ 40, otherwise **F**.

In [ ]:
# ---- 4. Scoring + scan_site ----------------------------------------------------
BAD_STATUSES = {"MISSING", "WEAK", "FAIL"}

def grade_from_score(score):
    for g, floor in (("A", 90), ("B", 75), ("C", 60), ("D", 40)):
        if score >= floor:
            return g
    return "F"

def scan_site(url):
    """Run the full passive audit on one URL and return a result dict."""
    if not url.startswith(("http://", "https://")):
        url = "https://" + url
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    result = dict(url=url, scanned_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
                  reachable=False, status_code=None, findings=[], score=None, grade="N/A")
    try:
        resp = session.get(url, timeout=TIMEOUT, stream=True, allow_redirects=True)
        resp.close()
    except requests.RequestException as e:
        result["error"] = str(e)
        return result

    result.update(reachable=True, status_code=resp.status_code, final_url=resp.url)
    findings = audit_headers(resp) + audit_cookies(resp) + audit_cors(session, resp.url)
    penalty = sum(SEVERITY_PENALTY[f["severity"]] for f in findings if f["status"] in BAD_STATUSES)
    result["findings"] = findings
    result["score"] = max(0, 100 - penalty)
    result["grade"] = grade_from_score(result["score"])
    return result

def findings_table(result):
    return pd.DataFrame(result["findings"])[["category", "check", "severity", "status", "detail"]]

print("Scanner ready.")

## 5. Safe local test lab
To demonstrate the scanner without touching third-party sites, this cell starts **two local servers** inside the notebook:

* `vulnerable` — deliberately misconfigured (no security headers, reflects any `Origin` with credentials, leaks versions, weak cookie).
* `hardened` — correctly configured with strong headers and no permissive CORS.

In [ ]:
# ---- 5. Local demo servers -----------------------------------------------------
class VulnerableHandler(BaseHTTPRequestHandler):
    server_version, sys_version = "Apache/2.4.49", ""
    def _cors(self):
        origin = self.headers.get("Origin")
        if origin:
            self.send_header("Access-Control-Allow-Origin", origin)      # reflected!
            self.send_header("Access-Control-Allow-Credentials", "true")
    def do_GET(self):
        self.send_response(200)
        self.send_header("Content-Type", "text/html")
        self.send_header("X-Powered-By", "PHP/7.4.3")
        self.send_header("Set-Cookie", "sessionid=abc123; Path=/")
        self._cors(); self.end_headers()
        self.wfile.write(b"<h1>Vulnerable demo</h1>")
    def do_OPTIONS(self):
        self.send_response(204)
        self._cors()
        self.send_header("Access-Control-Allow-Methods", "GET, POST, PUT, DELETE")
        self.end_headers()
    def log_message(self, *a): pass

class HardenedHandler(BaseHTTPRequestHandler):
    server_version, sys_version = "webserver", ""
    def do_GET(self):
        self.send_response(200)
        for k, v in {
            "Content-Type": "text/html",
            "Strict-Transport-Security": "max-age=31536000; includeSubDomains",
            "Content-Security-Policy": "default-src 'self'; frame-ancestors 'none'",
            "X-Frame-Options": "DENY",
            "X-Content-Type-Options": "nosniff",
            "Referrer-Policy": "strict-origin-when-cross-origin",
            "Permissions-Policy": "geolocation=(), camera=(), microphone=()",
            "Cross-Origin-Opener-Policy": "same-origin",
            "Cross-Origin-Resource-Policy": "same-origin",
            "Set-Cookie": "sessionid=abc123; Path=/; Secure; HttpOnly; SameSite=Strict",
        }.items():
            self.send_header(k, v)
        self.end_headers()
        self.wfile.write(b"<h1>Hardened demo</h1>")
    def do_OPTIONS(self):
        self.send_response(204); self.end_headers()
    def log_message(self, *a): pass

def start_server(handler):
    srv = HTTPServer(("127.0.0.1", 0), handler)         # port 0 = any free port
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    return srv, f"http://127.0.0.1:{srv.server_address[1]}/"

vuln_srv, VULN_URL = start_server(VulnerableHandler)
hard_srv, HARD_URL = start_server(HardenedHandler)
print("Vulnerable lab :", VULN_URL)
print("Hardened lab   :", HARD_URL)

## 6. Run the scans
Edit `TARGETS` to add your own sites (only ones you are authorised to test), for example `"https://your-own-site.com"`.

In [ ]:
# ---- 6. Run scans --------------------------------------------------------------
TARGETS = [VULN_URL, HARD_URL]
# TARGETS += ["https://your-own-site.com"]     # <- add only sites you are allowed to test

results = []
for t in TARGETS:
    print(f"Scanning {t} ...")
    results.append(scan_site(t))

summary = pd.DataFrame([{
    "URL": r["url"], "Reachable": r["reachable"], "Score": r["score"], "Grade": r["grade"],
    "High": sum(f["severity"] == "High" and f["status"] in BAD_STATUSES for f in r["findings"]),
    "Medium": sum(f["severity"] == "Medium" and f["status"] in BAD_STATUSES for f in r["findings"]),
    "Low": sum(f["severity"] == "Low" and f["status"] in BAD_STATUSES for f in r["findings"]),
} for r in results])
summary

In [ ]:
# ---- 6b. Detailed findings: vulnerable lab ------------------------------------
pd.set_option("display.max_colwidth", 90, "display.width", 200)
findings_table(results[0])

In [ ]:
# ---- 6c. Detailed findings: hardened lab --------------------------------------
findings_table(results[1])

## 7. Visualise the results

In [ ]:
# ---- 7. Chart ------------------------------------------------------------------
ok = [r for r in results if r["reachable"]]
names = {VULN_URL: "vulnerable lab", HARD_URL: "hardened lab"}
labels = [names.get(r["url"], urlparse(r["url"]).netloc) for r in ok]
scores = [r["score"] for r in ok]
colors = ["#d9534f" if s < 60 else "#f0ad4e" if s < 90 else "#5cb85c" for s in scores]

plt.figure(figsize=(6, 3.5))
bars = plt.bar(labels, scores, color=colors)
plt.ylim(0, 110); plt.ylabel("Security score (0-100)")
plt.title("Web Scanner - Security Score per Site")
for b, r in zip(bars, ok):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 2,
             f"{r['score']} ({r['grade']})", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "score_chart.png"), dpi=150)
plt.show()

## 8. Export reports (JSON, CSV, HTML)

In [ ]:
# ---- 8. Export -----------------------------------------------------------------
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# JSON (full detail)
json_path = os.path.join(REPORT_DIR, f"scan_{stamp}.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

# CSV (flat table)
csv_path = os.path.join(REPORT_DIR, f"scan_{stamp}.csv")
rows = [dict(url=r["url"], grade=r["grade"], score=r["score"], **fnd)
        for r in results for fnd in r["findings"]]
pd.DataFrame(rows).to_csv(csv_path, index=False)

# HTML (human-readable)
def to_html(results):
    out = ["<html><head><meta charset='utf-8'><title>Web Scan Report</title>",
           "<style>body{font-family:Arial;margin:30px}table{border-collapse:collapse;width:100%;margin-bottom:30px}",
           "td,th{border:1px solid #ccc;padding:6px;font-size:13px}th{background:#1f3a5f;color:#fff}",
           ".High{background:#f8d7da}.Medium{background:#fff3cd}.Low{background:#e2e3e5}</style></head><body>",
           "<h1>Web Security Scan Report</h1>"]
    for r in results:
        out.append(f"<h2>{html.escape(r['url'])} &mdash; Grade {r['grade']} (score {r['score']})</h2>")
        out.append("<table><tr><th>Category</th><th>Check</th><th>Severity</th><th>Status</th><th>Detail</th><th>Fix</th></tr>")
        for f in r["findings"]:
            cls = f["severity"] if f["status"] in BAD_STATUSES else ""
            out.append(f"<tr class='{cls}'>" + "".join(
                f"<td>{html.escape(str(f[k]))}</td>" for k in
                ("category", "check", "severity", "status", "detail", "fix")) + "</tr>")
        out.append("</table>")
    out.append("</body></html>")
    return "\n".join(out)

html_path = os.path.join(REPORT_DIR, f"scan_{stamp}.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write(to_html(results))

print("Saved:", json_path, csv_path, html_path, sep="\n  ")

In [ ]:
# ---- 9. Clean up demo servers ----------------------------------------------------
vuln_srv.shutdown(); hard_srv.shutdown()
print("Demo servers stopped.")

## 9. Conclusion
The scanner correctly separated the deliberately insecure server (low score, High-severity CORS failures, missing headers, leaky banners, weak cookie) from the hardened one (grade A, no findings). It can be pointed at any authorised site by adding its URL to `TARGETS`.

**Possible extensions:** scan a list of URLs from a file, add TLS certificate checks, run in CI as a security gate, and compare scans over time.